In [1]:
import numpy as np
import csv

# Given data specification
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ", "9", "12", "", "45", "182", "96"]
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

print("Libraries imported and dataset loaded successfully.")

Libraries imported and dataset loaded successfully.


In [2]:
def clean_machakos(raw_list):
    valid_vals = []
    parsed = []
    imputed_indices = []

    # First pass: parse valid integers and identify missing/invalid entries
    for i, val in enumerate(raw_list):
        try:
            num = int(val.strip())
            parsed.append(num)
            valid_vals.append(num)
        except ValueError:
            parsed.append(None)
            imputed_indices.append(i + 1) # 1-indexed month number

    # Calculate rounded average of valid Machakos values
    avg_val = round(sum(valid_vals) / len(valid_vals))

    # Replace None with the rounded average
    cleaned_machakos = [avg_val if p is None else p for p in parsed]

    print(f"Imputed months for Machakos (1-indexed): {imputed_indices}")
    print(f"Imputed replacement value (rounded average): {avg_val}")
    return cleaned_machakos

machakos_cleaned = clean_machakos(machakos_raw)
print("Machakos Cleaned Rainfall List:", machakos_cleaned)

Imputed months for Machakos (1-indexed): [4, 9]
Imputed replacement value (rounded average): 68
Machakos Cleaned Rainfall List: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


In [3]:
# Combine clean rows with cleaned Machakos row into a single list and convert to NumPy array
all_data = clean_rows + [machakos_cleaned]
rainfall_matrix = np.array(all_data, dtype=int)

print("--- Array Verification ---")
print("Shape:", rainfall_matrix.shape)
print("Data Type (dtype):", rainfall_matrix.dtype)

--- Array Verification ---
Shape: (5, 12)
Data Type (dtype): int64


In [4]:
# Task 3: Annual total rainfall per county and monthly average across all counties
annual_totals = np.sum(rainfall_matrix, axis=1)
monthly_averages = np.mean(rainfall_matrix, axis=0)

# Task 4: Wettest month per county using np.argmax and zip()
wettest_indices = np.argmax(rainfall_matrix, axis=1)
wettest_months = [month_names[idx] for idx in wettest_indices]

# Task 5: Count dry months (< 50 mm) using a boolean mask
dry_mask = rainfall_matrix < 50
dry_month_counts = np.sum(dry_mask, axis=1)

# Task 6: Two consecutive months with highest combined rainfall (shifted-array technique)
consecutive_sums = rainfall_matrix[:, :-1] + rainfall_matrix[:, 1:]
best_consecutive_idx = np.argmax(consecutive_sums, axis=1)
planting_windows = [f"{month_names[idx]}-{month_names[idx+1]}" for idx in best_consecutive_idx]

# Print summary results for verification
for c, tot, wet, dry, win in zip(counties, annual_totals, wettest_months, dry_month_counts, planting_windows):
    print(f"County: {c:<10} | Annual: {tot}mm | Wettest: {wet:<3} | Dry Months: {dry} | Window: {win}")

County: Nairobi    | Annual: 1045mm | Wettest: Apr | Dry Months: 4 | Window: Apr-May
County: Nakuru     | Annual: 916mm | Wettest: Apr | Dry Months: 3 | Window: Apr-May
County: Kisumu     | Annual: 1371mm | Wettest: Apr | Dry Months: 0 | Window: Apr-May
County: Mombasa    | Annual: 1075mm | Wettest: May | Dry Months: 2 | Window: Apr-May
County: Machakos   | Annual: 811mm | Wettest: Nov | Dry Months: 6 | Window: Nov-Dec


In [5]:
csv_filename = "rainfall_summary.csv"
header = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

with open(csv_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    for c, tot, wet, dry, win in zip(counties, annual_totals, wettest_months, dry_month_counts, planting_windows):
        writer.writerow([c, int(tot), wet, int(dry), win])

print(f"Successfully exported data to {csv_filename}.")

Successfully exported data to rainfall_summary.csv.


## Planting Advisory
Based on our multi-county rainfall distribution analysis, agricultural extension officers should advise farmers to align their planting schedules precisely with the identified two-consecutive-month peak windows to maximize crop germination. Regions experiencing higher counts of dry months below 50 mm will require targeted supplemental irrigation and water-harvesting infrastructure to mitigate mid-season drought risks. Furthermore, stakeholders must account for data-quality caveats, as missing station records in Machakos required imputation using rounded averages, which can smooth out extreme local weather anomalies. Incorporating these localized planting windows into NGO SMS advisories will directly improve seasonal agricultural yields and food security across all five target counties.